# Phase 3.1 Notebook: Entity Disambiguation
Reads candidates from data/20_canidate_generation and writes only to data/30_entity_disambiguation.
This phase is manual-only for final decisions.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'data').exists() and (cur / 'speakermining' / 'src').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'speakermining' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

In [ ]:
from process.entity_disambiguation.config import (
    INPUT_PHASE_DIR, PHASE_DIR, FILE_SCORES, FILE_SELECTIONS, SELECTION_COLUMNS
)
from process.entity_disambiguation.similarity import score_candidates
from process.entity_disambiguation.recommendation import build_recommendations
from process.entity_disambiguation.selection import (
    save_recommendations,
    build_review_sheet,
    require_manual_gate,
)

candidates = pd.read_csv(ROOT / INPUT_PHASE_DIR / 'candidates.csv')
scores = score_candidates(candidates)
recs = build_recommendations(scores, top_n=1)
review = build_review_sheet(candidates, scores, recs, top_k=5)

(ROOT / PHASE_DIR).mkdir(parents=True, exist_ok=True)
scores.to_csv(ROOT / PHASE_DIR / FILE_SCORES, index=False)
save_recommendations(recs, ROOT / PHASE_DIR)
review.to_csv(ROOT / PHASE_DIR / 'review_sheet.csv', index=False)

print(f'scores: {len(scores)}')
print(f'recommendations: {len(recs)}')
print(f'review rows: {len(review)}')

In [ ]:
selections_path = ROOT / PHASE_DIR / FILE_SELECTIONS
if not selections_path.exists():
    template = recs.rename(columns={'recommended_candidate_id': 'selected_candidate_id'})[[
        'mention_id', 'selected_candidate_id'
    ]].copy()
    template['decision'] = ''
    template['reason'] = ''
    template['reviewer'] = ''
    template['reviewed_at'] = ''
    template = template[SELECTION_COLUMNS]
    template.to_csv(selections_path, index=False)
    print(f'Created manual template: {selections_path}')
else:
    print(f'Manual decisions file exists: {selections_path}')

In [ ]:
try:
    decisions = require_manual_gate(ROOT / PHASE_DIR)
    print(f'Manual gate passed with {len(decisions)} decisions.')
except Exception as exc:
    print('Manual gate not passed yet. Fill selections.csv and rerun this cell.')
    print(str(exc))